# Medallion Pipeline — Bronze → Silver → Gold

A single notebook that runs the full medallion architecture for the MLE Supply Chain pipeline. Reads raw CSV from a UC Volume (Bronze), cleans and validates (Silver), and aggregates to the product-city-date grain (Gold). Uses OOP class-based design with Python `logging` for structured output. All parameters come from `dev.yaml`.

**Architecture:**

```
Bronze (raw CSV)  →  Silver (cleaned + DQ)  →  Gold (aggregated demand)  →  Feature Engineering
```

In [0]:
import yaml
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


class BaseLayer:
    """Base class for medallion layer ingestion with shared config and logging.

    Provides config loading from a YAML file, logger setup, and schema creation
    in Unity Catalog. Subclasses implement layer-specific read, transform,
    write, and validate logic.

    Attributes:
        config_path: Path to the YAML configuration file.
        layer_name: Name of the layer (used as the logger name).
        cfg: Parsed YAML configuration dictionary.
        catalog: Unity Catalog name from config.
    """

    def __init__(self, config_path: str, layer_name: str):
        self.config_path = config_path
        self.layer_name = layer_name
        self.logger = logging.getLogger(layer_name)
        self._load_config()

    def _load_config(self) -> None:
        """Load YAML configuration from the configured path."""
        with open(self.config_path, "r") as f:
            self.cfg = yaml.safe_load(f)
        self.catalog = self.cfg["catalog"]
        self.logger.info(f"Config loaded from {self.config_path}")

    def ensure_schema(self, schema_name: str) -> None:
        """Create the target schema in Unity Catalog if it doesn't exist."""
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{schema_name}")
        self.logger.info(f"Ensured schema exists: {self.catalog}.{schema_name}")

In [0]:
class BronzeIngestion(BaseLayer):
    """Bronze layer: read raw CSV from a UC Volume and write to a Bronze Delta table as-is.

    Adds a single audit column (bronze_ingestion_timestamp) to track when the
    data was ingested. All source format and write options come from the
    'bronze' section of dev.yaml.

    Attributes:
        full_table: Fully qualified Bronze table name (catalog.schema.table).
        source_path: Path to the source CSV file in a UC Volume.
    """

    def __init__(self, config_path: str = "/Workspace/Users/girija.zirange@bizmetric.com/MLE/config/dev.yaml"):
        super().__init__(config_path, "bronze_ingestion")
        self.layer_cfg = self.cfg["bronze"]
        self.schema = self.cfg["schemas"]["bronze"]
        self.source_path = self.cfg["source_path"]
        self.table_name = self.layer_cfg["target"]["table"]
        self.full_table = f"{self.catalog}.{self.schema}.{self.table_name}"
        self.logger.info(f"Source:  {self.source_path}")
        self.logger.info(f"Target:  {self.full_table}")

    def read_source(self):
        """Read the source file as-is using format and options from config."""
        reader = spark.read.format(self.layer_cfg["source"]["format"])
        for opt_key, opt_val in self.layer_cfg["source"]["options"].items():
            reader = reader.option(opt_key, opt_val)
        self.df = reader.load(self.source_path)
        self.row_count = self.df.count()
        self.logger.info(f"Rows read: {self.row_count:,}")
        self.logger.info(f"Columns:  {len(self.df.columns)}")
        return self.df

    def write_bronze(self) -> None:
        """Add an ingestion-timestamp audit column and write to the Bronze Delta table."""
        self.ensure_schema(self.schema)
        audit_col = self.layer_cfg["audit_columns"]["ingestion_timestamp"]
        write_cfg = self.layer_cfg["write"]
        (self.df
            .withColumn(audit_col, F.current_timestamp())
            .write
            .format(write_cfg["format"])
            .mode(write_cfg["mode"])
            .option("overwriteSchema", str(write_cfg["overwrite_schema"]).lower())
            .saveAsTable(self.full_table))
        self.logger.info(f"Written {self.row_count:,} rows to {self.full_table}")

    def validate(self) -> None:
        """Log Bronze table row and column counts."""
        bronze_df = spark.table(self.full_table)
        self.logger.info(f"Bronze table: {bronze_df.count():,} rows, {len(bronze_df.columns)} columns")

    def run(self) -> None:
        """Execute the full Bronze ingestion pipeline."""
        self.read_source()
        self.write_bronze()
        self.validate()

In [0]:
class SilverIngestion(BaseLayer):
    """Silver layer: deduplicate, convert types, validate DQ, and split good/bad records.

    Reads from the Bronze table, removes duplicates on transaction_id (keeping
    earliest ingestion), applies type conversions, runs data-quality checks,
    and writes good records to the Silver table and bad records to quarantine.

    Attributes:
        bronze_full: Fully qualified Bronze source table name.
        silver_full: Fully qualified Silver target table name.
        quarantine_full: Fully qualified quarantine table name.
    """

    def __init__(self, config_path: str = "/Workspace/Users/girija.zirange@bizmetric.com/MLE/config/dev.yaml"):
        super().__init__(config_path, "silver_ingestion")
        self.layer_cfg = self.cfg["silver"]
        self.bronze_schema = self.cfg["schemas"]["bronze"]
        self.silver_schema = self.cfg["schemas"]["silver"]
        self.bronze_table = self.layer_cfg["source"]["bronze_table"]
        self.silver_table = self.layer_cfg["target"]["silver_table"]
        self.quarantine_table = self.layer_cfg["target"]["quarantine_table"]
        self.bronze_full = f"{self.catalog}.{self.bronze_schema}.{self.bronze_table}"
        self.silver_full = f"{self.catalog}.{self.silver_schema}.{self.silver_table}"
        self.quarantine_full = f"{self.catalog}.{self.silver_schema}.{self.quarantine_table}"
        self.logger.info(f"Bronze source:     {self.bronze_full}")
        self.logger.info(f"Silver target:     {self.silver_full}")
        self.logger.info(f"Quarantine target: {self.quarantine_full}")

    def read_bronze(self):
        """Read the Bronze Delta table."""
        self.bronze_df = spark.table(self.bronze_full)
        self.bronze_count = self.bronze_df.count()
        self.logger.info(f"Bronze rows read: {self.bronze_count:,}")
        return self.bronze_df

    def deduplicate(self):
        """Deduplicate on the configured key column, keeping the earliest order-column value."""
        dedup_key = self.layer_cfg["dedup"]["key_column"]
        dedup_order = self.layer_cfg["dedup"]["order_column"]
        window = Window.partitionBy(dedup_key).orderBy(dedup_order)
        self.df_deduped = (
            self.bronze_df
            .withColumn("_row_rank", F.row_number().over(window))
            .filter(F.col("_row_rank") == 1)
            .drop("_row_rank")
        )
        self.dedup_count = self.df_deduped.count()
        self.removed_count = self.bronze_count - self.dedup_count
        self.logger.info(f"After dedup: {self.dedup_count:,} rows (removed {self.removed_count:,} duplicates)")
        return self.df_deduped

    def convert_types(self):
        """Apply type conversions from config and add the silver ingestion timestamp."""
        self.logger.info("Data type conversions applied:")
        df = self.df_deduped
        for conv in self.layer_cfg["type_conversions"]:
            col_name = conv["column"]
            to_type = conv["to_type"]
            if to_type == "date":
                df = df.withColumn(col_name, F.to_date(F.col(col_name)))
            elif to_type == "timestamp":
                df = df.withColumn(col_name, F.to_timestamp(F.col(col_name)))
            self.logger.info(f"  {col_name} -> {to_type}")
        audit_col = self.layer_cfg["audit_columns"]["silver_ingestion_timestamp"]
        df = df.withColumn(audit_col, F.current_timestamp())
        self.logger.info(f"  {audit_col} -> timestamp (new)")
        self.df_typed = df
        return self.df_typed

    def validate_dq(self):
        """Flag bad records based on DQ rules from config and split into good/bad DataFrames."""
        dq_cfg = self.layer_cfg["data_quality"]
        bad_conditions = []
        for col in dq_cfg["critical_columns"]:
            bad_conditions.append(F.col(col).isNull())
        for col in dq_cfg["date_validation_columns"]:
            bad_conditions.append(F.col(col).isNull())
        for rule in dq_cfg.get("format_validations", []):
            bad_conditions.append(
                F.col(rule["column"]).isNotNull() & ~F.col(rule["column"]).rlike(rule["pattern"])
            )
        for rule in dq_cfg.get("cross_field_validations", []):
            expr = rule["expression"]
            rule_cols = rule["columns"]
            null_cond = F.lit(False)
            for c in rule_cols:
                null_cond = null_cond | F.col(c).isNull()
            bad_conditions.append(null_cond | ~F.expr(expr))
        is_bad = bad_conditions[0]
        for cond in bad_conditions[1:]:
            is_bad = is_bad | cond
        self.df_typed = self.df_typed.withColumn("_is_bad_record", is_bad)
        self.df_good = self.df_typed.filter(~F.col("_is_bad_record")).drop("_is_bad_record")
        df_bad = self.df_typed.filter(F.col("_is_bad_record"))
        # Build quarantine reasons
        reason_whens = []
        for item in self.layer_cfg["quarantine"]["reason_labels"]:
            if "column" in item:
                reason_whens.append(F.when(F.col(item["column"]).isNull(), F.lit(item["label"])))
            elif "validation" in item:
                for rule in dq_cfg.get("cross_field_validations", []):
                    if rule["name"] == item["validation"]:
                        expr = rule["expression"]
                        rule_cols = rule["columns"]
                        null_cond = F.lit(False)
                        for c in rule_cols:
                            null_cond = null_cond | F.col(c).isNull()
                        reason_whens.append(
                            F.when(null_cond | ~F.expr(expr), F.lit(item["label"]))
                        )
            elif "format_validation" in item:
                for rule in dq_cfg.get("format_validations", []):
                    if rule["name"] == item["format_validation"]:
                        reason_whens.append(
                            F.when(
                                F.col(rule["column"]).isNotNull()
                                & ~F.col(rule["column"]).rlike(rule["pattern"]),
                                F.lit(item["label"]),
                            )
                        )
        bad_reason_expr = F.concat_ws("; ", *reason_whens)
        self.df_bad = df_bad.withColumn("_quarantine_reason", bad_reason_expr).drop("_is_bad_record")
        self.good_count = self.df_good.count()
        self.bad_count = self.df_bad.count()
        self.logger.info(f"Good records:  {self.good_count:,}")
        self.logger.info(f"Bad records:   {self.bad_count:,}")
        return self.df_good, self.df_bad

    def write_silver(self) -> None:
        """Write good records to the Silver table and bad records to the Quarantine table."""
        self.ensure_schema(self.silver_schema)
        write_cfg = self.layer_cfg["write"]
        (self.df_good
            .write
            .format(write_cfg["format"])
            .mode(write_cfg["mode"])
            .option("overwriteSchema", str(write_cfg["overwrite_schema"]).lower())
            .saveAsTable(self.silver_full))
        self.logger.info(f"Written {self.good_count:,} good records to {self.silver_full}")
        if self.bad_count > 0:
            (self.df_bad
                .write
                .format(write_cfg["format"])
                .mode(write_cfg["mode"])
                .option("overwriteSchema", str(write_cfg["overwrite_schema"]).lower())
                .saveAsTable(self.quarantine_full))
            self.logger.info(f"Written {self.bad_count:,} bad records to {self.quarantine_full}")
        else:
            self.logger.info("No bad records to quarantine.")

    def validate(self) -> None:
        """Log Silver ingestion summary."""
        self.logger.info(f"Silver: {self.good_count:,} good, {self.bad_count:,} bad, {self.removed_count:,} duplicates removed")

    def run(self) -> None:
        """Execute the full Silver ingestion pipeline."""
        self.read_bronze()
        self.deduplicate()
        self.convert_types()
        self.validate_dq()
        self.write_silver()
        self.validate()

In [0]:
class GoldIngestion(BaseLayer):
    """Gold layer: aggregate Silver data to the product-city-date grain for demand analysis.

    Reads cleaned, validated transaction data from the Silver layer and
    aggregates to the grain of (product_name, destination_city, transaction_date).
    This Gold table serves as the business-ready foundation for downstream
    feature engineering and demand forecasting, without any ML-specific
    transformations.

    Attributes:
        silver_full: Fully qualified Silver source table name.
        gold_full: Fully qualified Gold target table name.
        grain_cols: List of columns defining the Gold grain.
    """

    def __init__(self, config_path: str = "/Workspace/Users/girija.zirange@bizmetric.com/MLE/config/dev.yaml"):
        super().__init__(config_path, "gold_ingestion")
        self.layer_cfg = self.cfg["gold"]
        self.silver_schema = self.cfg["schemas"]["silver"]
        self.gold_schema = self.cfg["schemas"]["gold"]
        self.silver_table = self.layer_cfg["source"]["silver_table"]
        self.gold_table = self.layer_cfg["target"]["gold_table"]
        self.silver_full = f"{self.catalog}.{self.silver_schema}.{self.silver_table}"
        self.gold_full = f"{self.catalog}.{self.gold_schema}.{self.gold_table}"
        self.grain_cols = self.layer_cfg["grain"]["key_columns"]
        self.logger.info(f"Silver source:  {self.silver_full}")
        self.logger.info(f"Gold target:    {self.gold_full}")

    def read_silver(self):
        """Read the Silver Delta table (cleaned, typed, deduplicated, validated)."""
        self.silver_df = spark.table(self.silver_full)
        self.silver_count = self.silver_df.count()
        self.logger.info(f"Silver rows read: {self.silver_count:,}")
        return self.silver_df

    def aggregate(self):
        """Aggregate Silver data to the Gold grain: (product_name, destination_city, transaction_date)."""
        self.logger.info("Aggregating Silver data to Gold grain...")
        self.gold_df = (
            self.silver_df
            .groupBy("product_name", "destination_city", "transaction_date")
            .agg(
                # --- Dimensions (functionally dependent on grain key; use MIN) ---
                F.min("transaction_month").alias("transaction_month"),
                F.min("product_category").alias("product_category"),
                F.min("quantity_unit").alias("quantity_unit"),
                # --- Target / demand measures ---
                F.sum("demand_quantity").alias("demand_quantity"),
                F.sum("ordered_quantity").alias("ordered_quantity"),
                F.countDistinct("transaction_id").alias("transaction_count"),
                # --- Business measures (averages) ---
                F.avg("unit_price_usd").alias("avg_unit_price_usd"),
                F.avg("product_cost_usd").alias("avg_product_cost_usd"),
                # --- Business measures (totals) ---
                F.sum("total_cost_usd").alias("total_cost_usd"),
                # --- Inventory / operational measures ---
                F.avg("available_inventory").alias("avg_available_inventory"),
                # --- Supplier / quality measures ---
                F.avg("supplier_reliability_score").alias("avg_supplier_reliability_score"),
            )
            .select(
                "transaction_date",
                "transaction_month",
                "product_name",
                "product_category",
                "quantity_unit",
                "destination_city",
                "demand_quantity",
                "ordered_quantity",
                "transaction_count",
                "avg_unit_price_usd",
                "avg_product_cost_usd",
                "total_cost_usd",
                "avg_available_inventory",
                "avg_supplier_reliability_score",
            )
        )
        self.gold_count = self.gold_df.count()
        self.logger.info(f"Silver: {self.silver_count:,} → Gold: {self.gold_count:,} ({self.silver_count / self.gold_count:.2f}x compression)")
        return self.gold_df

    def write_gold(self) -> None:
        """Write the Gold table as a Delta table (idempotent — overwrite mode)."""
        self.ensure_schema(self.gold_schema)
        write_cfg = self.layer_cfg["write"]
        (self.gold_df.write
            .format(write_cfg["format"])
            .mode(write_cfg["mode"])
            .option("overwriteSchema", str(write_cfg["overwrite_schema"]).lower())
            .saveAsTable(self.gold_full))
        self.logger.info(f"Written {self.gold_count:,} rows to {self.gold_full}")

    def validate(self) -> None:
        """Log Gold ingestion summary."""
        self.logger.info(f"Gold: {self.gold_count:,} rows from {self.silver_count:,} silver rows ({self.silver_count / self.gold_count:.2f}x compression)")

    def run(self) -> None:
        """Execute the full Gold ingestion pipeline."""
        self.read_silver()
        self.aggregate()
        self.write_gold()
        self.validate()

In [0]:
# Run the full medallion pipeline: Bronze → Silver → Gold
bronze = BronzeIngestion()
bronze.run()

silver = SilverIngestion()
silver.run()

gold = GoldIngestion()
gold.run()

logging.getLogger("pipeline").info("Medallion pipeline complete: Bronze → Silver → Gold")